# 02 敘述統計（講義 03a / 03b / 06b、HW02 / HW03）

In [ ]:
import math
import random
import statistics

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import stats

import statkit as sk
sk.setup()  # 字體大小 16、中文字型

In [ ]:
values = [178, 172, 175, 184, 172, 175, 165, 178, 177, 175,
          180, 182, 177, 183, 180, 178, 179, 162, 170, 171]   # 講義 03a：棒球隊身高

## 資料的尺度、母體與樣本（講義 01）

| 尺度 | 可以做什麼 | 例子 |
|---|---|---|
| 名目 nominal | 只能分類（相等或不相等） | 性別、地區代碼、**電話號碼、學號**、是否（0/1） |
| 順序 ordinal | 可以排序，但差距沒有意義 | 名次、滿意度 1–5、等級（低/中/高） |
| 區間 interval | 差距有意義，但 0 不代表「沒有」 | 溫度（°C）、年份 |
| 比例 ratio | 差距、倍數都有意義，0 代表沒有 | 身高、收入、時間、距離、次數 |

- 名目、順序合稱**質性（類別）**資料；區間、比例才是**量化**資料。代碼（例如 Region = 1–4）是類別，不要算平均數
- **母體 vs 樣本**、**參數 vs 統計量**：μ、σ 是母體的參數（固定但未知）；$\bar{x}$、$s$ 是樣本算出來的統計量（每次抽樣都不同，所以是隨機變數）

## 一次看全部：`sk.describe`（附每一項的標準寫法）

In [ ]:
sk.describe(values)

## 集中趨勢

In [ ]:
print('median        :', statistics.median(values))
print('mean          :', statistics.mean(values))
print('geometric mean:', round(statistics.geometric_mean(values), 2))
print('mode(s)       :', statistics.multimode(values))

## 離散程度：母體 vs 樣本

$\sigma^2 = \frac{1}{N}\sum (x_i-\mu)^2$（母體）、$s^2 = \frac{1}{n-1}\sum (x_i-\bar{x})^2$（樣本，Bessel 校正）

In [ ]:
print('population variance:', statistics.pvariance(values))
print('sample variance    :', round(statistics.variance(values), 2))
print('population std     :', round(statistics.pstdev(values), 2))
print('sample std         :', round(statistics.stdev(values), 2))

mean_value = statistics.mean(values)
mad = statistics.mean([abs(v - mean_value) for v in values])
print('MAD                :', mad)

# pandas：.var() / .std() 預設是樣本（ddof=1），母體要寫 ddof=0
s = pd.Series(values)
print(s.var(), s.var(ddof=0))

## 四分位數、百分位數

講義用 `statistics.quantiles`（exclusive 法）；pandas `.quantile()` 是 inclusive 法，**兩者可能不同**，題目沒指定就照講義。

In [ ]:
quartiles = statistics.quantiles(values, n=4)
print('Q1, Q2, Q3 (statistics):', quartiles)
print('Q1, Q3 (pandas)        :', s.quantile(0.25), s.quantile(0.75))

percentiles = statistics.quantiles(values, n=100)
print('90th percentile:', percentiles[89])

## 百分等級、離群值（1.5 IQR）

In [ ]:
sk.percent_rank(values, 184)
outliers = sk.outliers_iqr(values)

## z 分數

In [ ]:
z = [(v - statistics.mean(values)) / statistics.stdev(values) for v in values]
print([round(t, 2) for t in z])
print(sk.zscore(values).round(2).tolist())   # 同上（ddof=1）

## 用 z 分數找離群值（講義 03b）

$|z| = \left|\frac{x - \mu}{\sigma}\right| > 3$ 的值標為離群值（樣本資料用 $\bar{x}$、$s$）。另一種是 1.5 IQR 規則（上面）。

所有資料的 z 分數，**平均數一定是 0、標準差一定是 1**。

In [ ]:
data_with_outlier = pd.Series(values + [230])          # 多加一個極端值
z = (data_with_outlier - data_with_outlier.mean()) / data_with_outlier.std()
print(data_with_outlier[z.abs() > 3])
print('z 分數的平均數、標準差：', round(z.mean(), 4), round(z.std(), 4))

In [ ]:
sk.outliers_z(data_with_outlier)

## 柴比雪夫定理、經驗法則

至少 $1 - 1/k^2$ 落在 $\mu \pm k\sigma$；鐘形資料約 68% / 95% / 99.7%。

In [ ]:
values_mean = statistics.mean(values)
values_sd = statistics.pstdev(values)
for k in [1.5, 2]:
    lower, upper = values_mean - k * values_sd, values_mean + k * values_sd
    within = len([v for v in values if lower <= v <= upper]) / len(values)
    print(f'k = {k}: [{lower:.2f}, {upper:.2f}], {within * 100:.1f}% within, Chebyshev guarantees {(1 - 1 / k ** 2) * 100:.2f}%')

In [ ]:
display(sk.chebyshev(values))
display(sk.empirical_rule(values))

## 偏態、峰態（講義公式）

In [ ]:
mean_value = statistics.mean(values)
sd_value = statistics.stdev(values)
skewness = sum((v - mean_value) ** 3 for v in values) / len(values) / sd_value ** 3
kurtosis = sum((v - mean_value) ** 4 for v in values) / len(values) / sd_value ** 4
print('skewness:', round(skewness, 2), ' kurtosis:', round(kurtosis, 2))
# 平均數 < 中位數 → 傾向左偏；平均數 > 中位數 → 傾向右偏

## 共變異數、相關係數

In [ ]:
sizes = [75, 59, 85, 65, 72, 46, 107, 91, 75, 65, 88, 59]
prices = [315, 229, 355, 261, 234, 216, 308, 306, 289, 204, 265, 195]
cov = sum((x - statistics.mean(sizes)) * (y - statistics.mean(prices)) for x, y in zip(sizes, prices)) / len(sizes)
r = cov / (statistics.pstdev(sizes) * statistics.pstdev(prices))
print('population covariance:', round(cov, 2), ' r:', round(r, 3))
print('statistics.correlation:', round(statistics.correlation(sizes, prices), 3))
sk.cov_corr(sizes, prices)

## 多個變數一起分析（HW02 / HW03）

讀進 Excel 之後，常常要「依類別分組比較」或「幾個數值變數一起看」。

| 想做的事 | pandas 寫法 | statkit |
|---|---|---|
| 幾個數值欄並排比較 | `df[cols].agg([...])` | `sk.describe_cols(df, cols)` |
| 依類別分組比較 | `df.groupby('類別')['數值'].agg([...])` | `sk.describe_by(df, '數值', by='類別')` |
| 兩個類別交叉 | `df.pivot_table(...)`、`pd.crosstab(...)` | — |
| 相關 / 共變異數矩陣 | `df[cols].corr()`、`df[cols].cov()` | `sk.corr_matrix(df, cols)` |

VS Code snippets：`groupstat`、`pivot`、`crosstab`、`corrm`、`descm`、`skby`、`skcorr`。

In [ ]:
rng = np.random.default_rng(1)   # 範例資料：120 戶的地區、居住地、收入、負債、食物支出（考試時換成自己的 df）
n = 120
households = pd.DataFrame({
    'Region': rng.choice([1, 2, 3, 4], n),
    'Location': rng.choice([1, 2], n),
    'Income': rng.normal(60000, 15000, n).round(),
    'Debt': rng.normal(15000, 6000, n).clip(0).round(),
})
households['Food'] = (2000 + 0.06 * households['Income'] + rng.normal(0, 800, n)).round()
households['Region'] = households['Region'].map({1: 'North East', 2: 'Middle West', 3: 'South', 4: 'West'})
households.head()

### 幾個數值變數並排比較

pandas 的 `std` 是**樣本**標準差（ddof=1）、`quantile` 是 inclusive 算法；四分位數要照講義就用 `statistics.quantiles`。

In [ ]:
cols = ['Food', 'Income', 'Debt']
display(households[cols].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2))
for c in cols:
    print(c, 'Q1, Q2, Q3 =', [round(q, 2) for q in statistics.quantiles(households[c], n=4)])

In [ ]:
sk.describe_cols(households, cols)

### 依類別分組比較（groupby）

`groupby('類別欄')['數值欄']` 之後接 `.mean()`、`.sum()`、`.max()`，或用 `.agg([...])` 一次算好幾個。

In [ ]:
food_by_region = households.groupby('Region')['Food']
display(food_by_region.agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(2))
print(food_by_region.sum())   # 只要一個統計量

In [ ]:
sk.describe_by(households, 'Food', by='Region')

### 兩個類別變數交叉：樞紐分析表、交叉表

- `pivot_table`：兩個類別交叉後彙總一個數值（`aggfunc='mean'`、`'sum'`、`'count'`…）
- `pd.crosstab`：兩個類別的次數；`margins=True` 加合計、`normalize='index'` 看每一列的比例

In [ ]:
table = households.pivot_table(index='Region', columns='Location', values='Food', aggfunc='mean')
table.columns = table.columns.map({1: 'Metropolitan', 2: 'Non-metropolitan'})   # 1、2 換成名稱
display(table.round(2))
display(pd.crosstab(households['Region'], households['Location'], margins=True))
display(pd.crosstab(households['Region'], households['Location'], normalize='index').round(4))

### 相關係數、共變異數矩陣

`corr()` 一次算出兩兩之間的 r；`cov()` 預設是**樣本**共變異數（除以 n − 1），母體要寫 `cov(ddof=0)`。各組分開算相關係數：`groupby` 之後再算 `corr`。

In [ ]:
display(households[cols].corr().round(4))
display(households[cols].cov(ddof=0).round(2))   # 母體共變異數
print(households.groupby('Region')[['Income', 'Food']].apply(lambda g: g['Income'].corr(g['Food'])).round(4))

In [ ]:
sk.corr_matrix(households, cols)

畫圖（分組長條圖、依類別上色的散佈圖、散佈圖矩陣、分組盒鬚圖）見 `01_視覺化.ipynb` 的「多個變數一起畫」。

## 次數分配表、眾數組（HW03 1(a)）

In [ ]:
debt = pd.Series(stats.norm.rvs(loc=15000, scale=8000, size=200, random_state=1)).clip(lower=0)   # 範例資料

bins = range(0, 40000 + 5000, 5000)
freq = pd.cut(debt, bins=bins, right=False).value_counts(sort=False)
print(freq)
modal_class = freq.idxmax()
print('modal class:', modal_class, ' mode (midpoint):', modal_class.mid)

In [ ]:
sk.freq_table(debt, start=0, width=5000)

## 分組資料的平均數與變異數

In [ ]:
table = sk.freq_table(debt, start=0, width=5000)
sk.grouped_stats(table['組中點'], table['次數'], sample=True)

## 分組資料的中位數（講義 03b）

**不是用組中點**：找第 $\frac{N+1}{2}$ 項在哪一組（中位數組）、是該組的第 $r$ 項，然後在組內內插：

$$\text{中位數} = \text{下限} + \frac{r - 0.5}{f} \times \text{組距}$$

In [ ]:
edges = [160, 164, 168, 172, 176, 180, 184, 188]
freqs = [1, 1, 2, 5, 6, 4, 1]
position = (sum(freqs) + 1) / 2              # 第 10.5 項
cum = 0
for i, f in enumerate(freqs):
    if cum + f >= position:                   # 中位數組
        r = position - cum                    # 在這組裡是第幾項
        median = edges[i] + (r - 0.5) / f * (edges[i + 1] - edges[i])
        break
    cum += f
print(round(median, 2))                       # 講義：176.67

In [ ]:
sk.grouped_median(edges, freqs)

## 敘述統計的陷阱（講義 03c）

- **沒看分母**：「臨時取消的人大多是 50 幾歲」可能只是 50 幾歲的客人本來就多 → 要比**比例**（取消人數 ÷ 該年齡層人數），不是次數
- **倖存者偏誤**：只看得到「活下來」的資料（例如飛回來的轟炸機彈孔），結論會反過來
- **相關 ≠ 因果**：兩個變數相關，可能是第三個因素造成的（維度缺失）。最好的證明是**隨機實驗**；做不到時，至少**控制**其他因素
- **控制變因**：用雙因子樞紐分析，在每一個「其他因素」的層級裡分別比較，結論可能跟只看一個因子時不同

In [ ]:
# 自編資料：有沒有子女 vs 績效好壞；年資可能是背後的因素
staff = pd.DataFrame({
    'seniority': ['junior'] * 6 + ['senior'] * 6,
    'children':  ['no', 'no', 'no', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'no', 'yes'],
    'good':      [0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1],
})
display(staff.pivot_table(index='children', values='good', aggfunc='mean'))                          # 只看一個因子
display(staff.pivot_table(index='seniority', columns='children', values='good', aggfunc='mean'))     # 控制年資

## DataFrame 常用操作（讀 Excel 後）

In [ ]:
df = pd.DataFrame({'Food': [3000, 4200, 5100, 3900], 'Income': [40000, 52000, 61000, 45000], 'Region': [1, 2, 1, 3]})
display(df.describe())
display(df.corr())
display(df.groupby('Region')['Income'].agg(['count', 'mean', 'std']))
display(df[df['Income'] > 45000])
display(df.sort_values('Food', ascending=False))
# df = pd.read_excel('../data/xxx.xlsx', sheet_name='Consumer Food')